# Next-Horizon Architectural Evaluation: BROS 2D Relative Spatial Attention & Template-Adversarial Learning (DANN)
## Resolving Generalization Collapse in Visually-Rich Key-Value Extraction

**IE 643 Course Project** — Cross-Template Key-Value Field Extraction from Scanned Forms  
**Authors:** Team Unemployed and Unsupervised (Param Mehta 23b2439, Arjoe Basak 23b1295) · IIT Bombay  

---

### 1. Research Motivation & The Spatial Inductive Bias Dilemma
Our previous empirical benchmark on VRDU Registration Forms under matched sample size ($N=200$) revealed the **Spatial Inductive Bias Dilemma**:
1. **Baseline LiLT (Coordinate Transformers):** Suffered a **27.31-point / 29.8% relative generalization collapse** (91.56% Seen MTL -> 64.24% Unseen UTL) due to rote coordinate memorization of the $[0, 1000]^4$ absolute bounding box grid.
2. **RSA-GNN (Geometric Graph Transformers):** Suffered an even worse **43.92-point / 51.1% collapse** (85.97% -> 42.05%) because rigid planar Delaunay triangulation hardcodes local spatial edges. When form layouts shift from horizontal rows to vertical table grids, geometric message-passing scrambles across incorrect neighbors.

---

### 2. The Two Proposed Next-Generation Horizons

#### Horizon 1: BROS-Style 2D Relative Positional Attention (2D RoPE)
- Replaces absolute bounding box coordinates $[x_0, y_0, x_1, y_1]$ with **pairwise relative spatial displacements** $\Delta p_{ij} = p_i - p_j$.
- Displacements are embedded via 2D sinusoidal representations and injected directly into the multi-head self-attention matrix:
  $$\text{AttentionScores}_{ij} = \frac{q_i^T k_j}{\sqrt{d}} + q_i^T \text{PE}(\Delta p_{ij})$$
- **Key Property:** Absolute position on the page is discarded; self-attention relies exclusively on invariant pairwise spatial topology.

#### Horizon 2: Template-Adversarial Representation Learning (DANN)
- Connects a **Template Discriminator Head** ($D(h) \rightarrow \hat{y}_{\text{template}}$) to the document encoder via a **Gradient Reversal Layer (GRL)**.
- Dynamic adaptation schedule $\alpha(p) = \frac{2}{1 + \exp(-\gamma \cdot p)} - 1$.
- Total Objective: $\mathcal{L} = \mathcal{L}_{\text{task}} + \lambda_{\text{domain}} \mathcal{L}_{\text{domain}}$.
- During backprop, the discriminator minimizes template cross-entropy while the backbone receives reversed gradients, **mathematically enforcing template-invariant representations**.

--- 
## 0 — Environment Setup, Dual-GPU Check & Repo Clone

In [ ]:
import os, subprocess, sys

GITHUB_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
    print("[+] Successfully retrieved GITHUB_TOKEN from Kaggle Secrets.")
except Exception as e:
    print(f"[*] Notice: GITHUB_TOKEN not found in Kaggle Secrets ({e}). Defaulting to public HTTPS.")

REPO_NAME = "ABasakEE/KeyValue-Extraction"
REPO_DIR = "/kaggle/working/KeyValue-Extraction"
BRANCH_NAME = "feat/bros-relative-layout"

if not os.path.exists(REPO_DIR):
    print(f"[*] Cloning {REPO_NAME} (branch: {BRANCH_NAME})...")
    if GITHUB_TOKEN:
        clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_NAME}.git"
    else:
        clone_url = f"https://github.com/{REPO_NAME}.git"
    subprocess.run(["git", "clone", "-b", BRANCH_NAME, clone_url, REPO_DIR], check=True)
else:
    print(f"[*] Pulling latest commits on {BRANCH_NAME}...")
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH_NAME], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
os.chdir(REPO_DIR)
print(f"[+] Current working directory: {os.getcwd()}")

# Verify Dual-GPU hardware
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv

--- 
## 1 — Install Dependencies & Verify BROS / DANN Modules

In [ ]:
!pip install -q seqeval python-Levenshtein pyyaml

# Run comprehensive unit test suites to guarantee all components are operational
!python tests/test_bros_pipeline.py
!python tests/test_dann.py

--- 
## 2 — Clone Google VRDU Benchmark Dataset
Ensures `data/vrdu/registration-form` and `data/vrdu/ad-buy-form` are available.

In [ ]:
# Clone Google VRDU dataset if not already present
import os, subprocess

vrdu_dir = "data/vrdu"
if not os.path.exists(os.path.join(vrdu_dir, "registration-form", "main", "dataset.jsonl.gz")):
    print("[*] Cloning Google VRDU dataset from https://github.com/google-research-datasets/vrdu.git...")
    os.makedirs("data", exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/google-research-datasets/vrdu.git", vrdu_dir], check=True)
    print("[+] VRDU dataset cloned successfully.")
else:
    print("[+] VRDU dataset already present.")

# Verify both subcorpora exist
for sub in ["registration-form", "ad-buy-form"]:
    sub_p = os.path.join(vrdu_dir, sub, "main", "dataset.jsonl.gz")
    print(f"Checking {sub}: exists = {os.path.exists(sub_p)}")

---
## 3 — Smoke Test Suite for All Experiments & Ablations
Runs a fast 1-step dry run across all 4 experimental configurations to verify model construction, bounding box normalization, collation, forward pass, gradient backward pass, and inference soft-voting before launching long GPU training runs.

In [ ]:
# Execute smoke test suite across all 4 experimental configurations
!python src/smoke_test.py

--- 
## 4 — Experiment 1: BROS Relative Spatial Positional Attention Baseline
Evaluates standard BROS 2D relative spatial attention on VRDU Registration Forms under the matched ($N=200$) protocol.

In [ ]:
# Run BROS baseline with dynamic dual-GPU parallel dispatch
!python src/train.py --config configs/vrdu_bros.yaml --protocol matched --no-save-weights

--- 
## 5 — Experiment 2: BROS + Multi-Page Sliding Window & Coordinate Jitter
Combines BROS relative layout with 384-token sliding window soft-voting (eliminating the 83.3% truncation drop) and coordinate jitter.

In [ ]:
!python src/train.py --config configs/vrdu_track1_bros.yaml --protocol matched --no-save-weights

--- 
## 6 — Experiment 3: Template-Adversarial Representation Learning (DANN-LiLT)
Applies Gradient Reversal Layer and Template Discriminator to LiLT to mathematically force template-invariant representations.

In [ ]:
!python src/train.py --config configs/vrdu_dann_lilt.yaml --protocol matched --no-save-weights

--- 
## 7 — Experiment 4: Unified DANN-BROS (Relative Layout + Template Invariance)
The ultimate architectural synthesis: pairing 2D relative spatial displacements with adversarial template regularization.

In [ ]:
!python src/train.py --config configs/vrdu_dann_bros.yaml --protocol matched --no-save-weights

--- 
## 8 — Master Comparative Benchmark & Generalization Gap Report

In [ ]:
import json, glob
from pathlib import Path
import pandas as pd

def evaluate_matched_results(results_dir: str):
    mtl_files = glob.glob(f"{results_dir}/MTL_fold*/metrics.json")
    utl_files = glob.glob(f"{results_dir}/UTL_fold*/metrics.json")
    
    def get_mean(files):
        scores = []
        for f in files:
            data = json.loads(Path(f).read_text(encoding='utf-8'))
            scores.append(data.get('mean_f1', 0.0))
        return sum(scores) / len(scores) if scores else 0.0
    
    mtl_f1 = get_mean(mtl_files)
    utl_f1 = get_mean(utl_files)
    gap = mtl_f1 - utl_f1
    rel_drop = (gap / mtl_f1 * 100) if mtl_f1 > 0 else 0.0
    return mtl_f1, utl_f1, gap, rel_drop

experiments = [
    ("1. LiLT Baseline (Standard Coordinates)", "results/vrdu_track1_lilt"),
    ("2. Standalone RSA-GNN (Delaunay 16-D)", "results/gnn_matched"),
    ("3. BROS Baseline (2D Relative Positional Attention)", "results/vrdu_bros"),
    ("4. BROS + Window Soft-Voting + Jitter", "results/vrdu_track1_bros"),
    ("5. DANN-LiLT (Template-Adversarial Regularization)", "results/vrdu_dann_lilt"),
    ("6. DANN-BROS (Relative Layout + Adversarial)", "results/vrdu_dann_bros"),
]

rows = []
for name, p in experiments:
    if os.path.exists(p):
        mtl, utl, gap, rel = evaluate_matched_results(p)
        rows.append({
            "Architecture": name,
            "Seen (MTL F1)": f"{mtl*100:.2f}%",
            "Unseen (UTL F1)": f"{utl*100:.2f}%",
            "Generalization Gap (Δ)": f"{gap*100:+.2f} pts",
            "Relative Drop (%)": f"{rel:.1f}%"
        })

df = pd.DataFrame(rows)
print(df.to_markdown(index=False))
